In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys

!pip install facenet-pytorch

import os
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

In [4]:
import os
import torch
import numpy as np
from facenet_pytorch import MTCNN
from PIL import Image, ImageOps
from tqdm import tqdm

def process_dataset_aligned(base_input_path, dest_output_path,dataset_path = "galleries"):
    """
    Scorre le cartelle, allinea il volto basandosi sugli occhi e salva il crop.
    """

    # 1. Configurazione
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f'Running on device: {device}')

    # Inizializzazione MTCNN
    # Nota: post_process=False qui non influenza il salvataggio diretto su file,
    # ma è utile se volessimo i tensor normalizzati.
    mtcnn = MTCNN(
        keep_all=False,
        select_largest=True,
        margin=20,
        post_process=False,
        device=device
    )

    dataset_path = os.path.join(base_input_path, dataset_path)
    if not os.path.exists(dataset_path):
        print(f"Errore: La cartella {dataset_path} non esiste.")
        return

    subjects = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    print(f"Trovati {len(subjects)} soggetti. Inizio elaborazione con allineamento...")

    # Funzione Helper interna per gestire la logica di allineamento
    def align_and_save(img_path, save_full_path, detector):
        try:
            # A. Caricamento e EXIF Fix (cruciale per foto da smartphone)
            img = Image.open(img_path)
            img = ImageOps.exif_transpose(img)
            img = img.convert('RGB')

            # B. Detection Landmarks (solo coordinate, niente crop ancora)
            boxes, probs, landmarks = detector.detect(img, landmarks=True)

            # Se non trova facce o landmarks, saltiamo (o potremmo fare un fallback)
            if boxes is None or landmarks is None:
                return False

            # C. Calcolo Angolo di Rotazione
            # landmarks[0] è la faccia principale (perché select_largest=True)
            # punti: [occhio_sx, occhio_dx, naso, bocca_sx, bocca_dx]
            left_eye = landmarks[0][0]
            right_eye = landmarks[0][1]

            delta_x = right_eye[0] - left_eye[0]
            delta_y = right_eye[1] - left_eye[1]
            angle = np.degrees(np.arctan2(delta_y, delta_x))

            # D. Rotazione e Salvataggio finale
            # Se l'angolo è significativo (>1 grado), ruotiamo
            if abs(angle) > 1:
                # expand=True evita di tagliare angoli ruotando
                img_rotated = img.rotate(angle, resample=Image.BICUBIC, expand=True)

                # Rieseguiamo MTCNN sull'immagine raddrizzata per il crop finale e salvataggio
                detector(img_rotated, save_path=save_full_path)
            else:
                # Se è già dritta, usiamo l'originale
                detector(img, save_path=save_full_path)

            return True

        except Exception as e:
            # print(f"Errore allineamento {img_path}: {e}") # Debug
            return False

    # 2. Iterazione principale
    for subject in tqdm(subjects, desc="Soggetti"):
        subject_path = os.path.join(dataset_path, subject)

        # Cartella destinazione appiattita
        save_dir = os.path.join(dest_output_path, subject)
        os.makedirs(save_dir, exist_ok=True)

        for root, dirs, files in os.walk(subject_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    file_path = os.path.join(root, file)

                    # Nome univoco
                    parent_folder = os.path.basename(root)
                    unique_filename = f"{file}"
                    save_path = os.path.join(save_dir, unique_filename)

                    # Chiamata alla funzione di allineamento
                    align_and_save(file_path, save_path, mtcnn)

    print("\nElaborazione completata!")


INPUT_PATH = '/content/drive/MyDrive/BBA_Dataset'
OUTPUT_PATH = '/content/drive/MyDrive/BBA_Dataset_training'
DATASET_PATH = 'probes'

process_dataset_aligned(INPUT_PATH, OUTPUT_PATH, DATASET_PATH)

Running on device: cuda:0
Trovati 19 soggetti. Inizio elaborazione con allineamento...


Soggetti: 100%|██████████| 19/19 [14:40<00:00, 46.36s/it]


Elaborazione completata!


In [9]:
import os
import torch
import numpy as np
import pandas as pd
from facenet_pytorch import MTCNN
from PIL import Image, ImageOps
from tqdm import tqdm
import re

class FacePipeline:
    def __init__(self, base_input_root, base_output_root):
        """
        base_input_root: La cartella radice che contiene 'galleries', 'probes', ecc.
        base_output_root: Dove creare la struttura speculare di output.
        """
        self.base_input_root = base_input_root
        self.base_output_root = base_output_root

        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"Pipeline inizializzata su device: {self.device}")

        # Inizializza MTCNN
        self.mtcnn = MTCNN(
            keep_all=False,
            select_largest=True,
            margin=20,
            post_process=False,
            device=self.device
        )

    def _extract_label(self, subject_name):
        """
        Cerca di estrarre l'ID numerico dal nome della cartella.
        Esempio: 'subject_05' -> 5
        Garantisce che lo stesso soggetto abbia la stessa label in 'galleries' e 'probes'.
        """
        # Cerca numeri nel nome della cartella
        match = re.search(r'\d+', subject_name)
        if match:
            return int(match.group())
        else:
            # Fallback: Hash del nome o gestione custom se i nomi non hanno numeri
            print(f"Attenzione: Nessun numero trovato in '{subject_name}'. Assegno label provvisoria -1.")
            return -1

    def _align_and_save(self, img_path, save_full_path):
        """Logica di allineamento e crop (uguale a prima)."""
        try:
            img = Image.open(img_path)
            img = ImageOps.exif_transpose(img)
            img = img.convert('RGB')

            boxes, probs, landmarks = self.mtcnn.detect(img, landmarks=True)

            if boxes is None or landmarks is None:
                return False

            left_eye = landmarks[0][0]
            right_eye = landmarks[0][1]
            delta_x = right_eye[0] - left_eye[0]
            delta_y = right_eye[1] - left_eye[1]
            angle = np.degrees(np.arctan2(delta_y, delta_x))

            if abs(angle) > 1:
                img_rotated = img.rotate(angle, resample=Image.BICUBIC, expand=True)
                self.mtcnn(img_rotated, save_path=save_full_path)
            else:
                self.mtcnn(img, save_path=save_full_path)
            return True
        except Exception as e:
            return False

    def process_split(self, split_name):
        """
        Processa una specifica sottocartella (es. 'galleries' o 'probes').
        """
        input_split_path = os.path.join(self.base_input_root, split_name)
        output_split_path = os.path.join(self.base_output_root, split_name)

        if not os.path.exists(input_split_path):
            print(f"ERRORE: La cartella '{split_name}' non esiste in {self.base_input_root}")
            return

        print(f"\n--- Inizio elaborazione: {split_name.upper()} ---")

        dataset_records = []

        # Lista soggetti
        subjects = sorted([d for d in os.listdir(input_split_path) if os.path.isdir(os.path.join(input_split_path, d))])

        for subject_name in tqdm(subjects, desc=f"Soggetti in {split_name}"):

            # 1. Calcolo Label Coerente
            label = self._extract_label(subject_name)

            # Percorsi
            subject_input_dir = os.path.join(input_split_path, subject_name)
            subject_output_dir = os.path.join(output_split_path, subject_name)
            os.makedirs(subject_output_dir, exist_ok=True)

            # 2. Scansione file
            for root, _, files in os.walk(subject_input_dir):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):

                        file_path = os.path.join(root, file)
                        parent_name = os.path.basename(root) # es. 'recording_01'

                        # Nome file univoco
                        unique_filename = f"{file}"
                        save_full_path = os.path.join(subject_output_dir, unique_filename)

                        # 3. Processamento
                        success = self._align_and_save(file_path, save_full_path)

                        if success:
                            dataset_records.append({
                                'image_path': save_full_path,
                                'label': label,
                                'subject_name': subject_name,
                                'split': split_name # Utile per filtrare dopo
                            })

        # 4. Salvataggio CSV specifico per questo split
        if dataset_records:
            df = pd.DataFrame(dataset_records)
            csv_name = f"manifest_{split_name}.csv"
            csv_path = os.path.join(self.base_output_root, csv_name)
            df.to_csv(csv_path, index=False)
            print(f"Salvato manifest per {split_name} in: {csv_path}")
        else:
            print(f"Nessuna immagine valida trovata in {split_name}")

In [10]:
INPUT_ROOT = '/content/drive/MyDrive/BBA_Dataset'
OUTPUT_ROOT = '/content/drive/MyDrive/BBA_Dataset_processed'

pipeline = FacePipeline(INPUT_ROOT, OUTPUT_ROOT)

# 1. Processa Galleries
pipeline.process_split("galleries")

# 2. Processa Probes
pipeline.process_split("probes")

Pipeline inizializzata su device: cuda:0

--- Inizio elaborazione: GALLERIES ---


Soggetti in galleries: 100%|██████████| 19/19 [03:29<00:00, 11.01s/it]


Salvato manifest per galleries in: /content/drive/MyDrive/BBA_Dataset_processed/manifest_galleries.csv

--- Inizio elaborazione: PROBES ---


Soggetti in probes: 100%|██████████| 19/19 [07:51<00:00, 24.84s/it]

Salvato manifest per probes in: /content/drive/MyDrive/BBA_Dataset_processed/manifest_probes.csv
